# 10x Visium 空間トランスクリプトーム解析

このノートブックでは 10x Genomics Visium データの読み込み、QC、正規化、クラスタリング、空間可視化の基本フローを実行します。

## 必要なデータ
- `filtered_feature_bc_matrix/`（または H5）
- `spatial/`（positions, scale_factors, 画像）
- サンプルは `data/raw/<sample_id>/` に配置

## 1. 環境・パス設定

In [ ]:
import sys
from pathlib import Path

# プロジェクトルート（リポジトリ直下で jupyter を起動するか、notebooks/ から実行するかで自動判定）
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import get_paths, get_qc_config
from src.preprocessing import (
    add_qc_metrics,
    filter_cells_and_genes,
    normalize_and_hvg,
    read_visium_sample,
)
from src.clustering import build_spatial_neighbors, run_moran_i

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor="white")

paths = get_paths(PROJECT_ROOT)
qc_cfg = get_qc_config()

# サンプル名に合わせて変更
SAMPLE_ID = "sample1"
sample_path = paths["data_raw"] / SAMPLE_ID

## 2. Visium データの読み込み

In [ ]:
adata = read_visium_sample(sample_path)

print(adata)
print("Spatial coordinates (obsm['spatial']):", adata.obsm["spatial"].shape)

## 3. QC とフィルタリング

In [ ]:
add_qc_metrics(adata)
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4, multi_panel=True)

In [ ]:
adata = filter_cells_and_genes(
    adata,
    min_genes=qc_cfg["min_genes"],
    min_cells=qc_cfg["min_cells"],
    pct_counts_mt_max=qc_cfg["pct_counts_mt_max"],
)
print(adata)

## 4. 正規化とログ変換

In [ ]:
adata = normalize_and_hvg(adata, n_top_genes=qc_cfg["n_top_genes"])

## 5. スケールと PCA・クラスタリング

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, n_pcs=20)

sc.pp.neighbors(adata, n_neighbors=qc_cfg["n_neighbors"], n_pcs=qc_cfg["n_pcs"])
sc.tl.leiden(adata, resolution=qc_cfg["leiden_resolution"])
sc.tl.umap(adata)

## 6. 空間・UMAP 可視化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.umap(adata, color="leiden", ax=axes[0], show=False)
sc.pl.spatial(adata, color="leiden", ax=axes[1], show=False)
plt.tight_layout()
plt.show()

## 7. 遺伝子発現の空間プロット

In [ ]:
genes = adata.var_names[:4].tolist()  # または ['GENE1', 'GENE2', ...]
sc.pl.spatial(adata, color=genes, cmap="viridis", ncols=2)

## 8. Squidpy: 空間隣接とモランI

In [ ]:
build_spatial_neighbors(adata, coord_type="generic", delaunay=True)
run_moran_i(adata, n_genes=100)

adata.uns["moranI"].sort_values("I", ascending=False).head(10)

## 9. 結果の保存

In [ ]:
out_dir = paths["data_processed"]
out_dir.mkdir(parents=True, exist_ok=True)
adata.write(out_dir / f"{SAMPLE_ID}_processed.h5ad")